Joey Cahill

In [4]:
import os
import re
import time
import numpy as np
import pandas as pd
import requests

FILE_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_COMBINED.csv"
CHECKPOINT_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\geocode_checkpoint_opencage.csv"

OPENCAGE_API_KEY = os.getenv("OPENCAGE_API_KEY", "")
OPENCAGE_URL = "https://api.opencagedata.com/geocode/v1/json"

SLEEP_SECONDS = 1.0
N_BATCH = 1000
SAVE_EVERY = 50
CONFIDENCE_REVIEW_THRESHOLD = 6
PROVINCE_COL = "PROVINCE"
ID_COL = "PHARMACY_ID"

if not OPENCAGE_API_KEY:
    raise ValueError("OPENCAGE_API_KEY not found in environment.")

# ---------- helpers ----------
def clean_address(addr):
    if pd.isna(addr):
        return ""
    addr = str(addr)
    addr = re.sub(r"\bP\.?\s*O\.?\s*Box\b.*", "", addr, flags=re.I)
    addr = re.sub(r"\bC\/O\b.*", "", addr, flags=re.I)
    addr = re.sub(r"\b(near|opp|opposite|next to|behind)\b.*", "", addr, flags=re.I)
    addr = re.sub(r"\+?\d[\d\-\s]{7,}\d", "", addr)
    addr = addr.replace(";", ",")
    addr = re.sub(r"\s+", " ", addr).strip(" ,.-")
    return addr

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def best_locality(row):
    for col in ["CITY", "LOCAL_MUNICIPALITY", "METROPOLITAN_MUNICIPALITY", "DISTRICT_MUNICIPALITY"]:
        if col in row.index:
            v = row.get(col)
            if pd.notna(v):
                s = clean_text(v)
                if s and s.lower() != "nan":
                    return s
    return ""

def build_query(row):
    name = clean_text(row.get("PRACTICE_NAME"))
    addr = clean_address(row.get("ADDRESS"))
    loc  = best_locality(row)
    prov = clean_text(row.get(PROVINCE_COL))

    parts = []
    if addr: parts.append(addr)
    if loc:  parts.append(loc)
    if prov: parts.append(prov)
    parts.append("South Africa")

    q = ", ".join([p for p in parts if p])

    if len(addr) < 8 and name:
        q = ", ".join([name, q])

    return q

def opencage_geocode(query):
    params = {
        "q": query,
        "key": OPENCAGE_API_KEY,
        "limit": 1,
        "no_annotations": 0,
        "language": "en",
        "countrycode": "za",
    }
    r = requests.get(OPENCAGE_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    if data.get("total_results", 0) < 1:
        return None
    return data["results"][0]

def extract_fields(result):
    geometry = result.get("geometry", {}) or {}
    components = result.get("components", {}) or {}
    conf = result.get("confidence", None)
    return {
        "GEOCODE_LAT": geometry.get("lat"),
        "GEOCODE_LON": geometry.get("lng"),
        "GEOCODE_CONFIDENCE": conf,
        "GEOCODE_FORMATTED": result.get("formatted"),
        "GEOCODE_STATE": components.get("state"),
        "GEOCODE_CITY": components.get("city") or components.get("town") or components.get("village"),
        "GEOCODE_SUBURB": components.get("suburb"),
        "GEOCODE_ROAD": components.get("road"),
        "GEOCODE_HOUSENUMBER": components.get("house_number"),
    }

def province_mismatch(input_prov, geocoded_state):
    a = clean_text(input_prov).lower()
    b = clean_text(geocoded_state).lower()
    if not a or not b:
        return False
    return not (a in b or b in a)

# ---------- load CSV with safe fallback ----------
try:
    df = pd.read_csv(FILE_PATH)
except OSError:
    df = pd.read_csv(FILE_PATH, engine="python", encoding_errors="replace")

# ensure id is numeric-ish
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce")

# focus on missing coords
df["COORDS"] = df["COORDS"].replace({np.nan: None})
to_geocode = df[df["COORDS"].isna()].copy()


# ---------- load checkpoint ----------
if os.path.exists(CHECKPOINT_PATH):
    ck = pd.read_csv(CHECKPOINT_PATH)
    ck_ids = set(pd.to_numeric(ck[ID_COL], errors="coerce").dropna().astype(int).tolist())
    print(f"Checkpoint found. Already processed: {len(ck_ids)}")
else:
    ck = pd.DataFrame()
    ck_ids = set()
    print("No checkpoint found. Starting fresh.")

# deterministic first 1k
# sort entire missing list
to_geocode = to_geocode.sort_values(ID_COL).reset_index(drop=True)

# remove already processed IDs
to_geocode = to_geocode[~to_geocode[ID_COL].isin(ck_ids)]

# take next batch
to_geocode = to_geocode.head(N_BATCH).reset_index(drop=True)

buffer = []
new_done = 0

for idx, row in enumerate(to_geocode.to_dict("records"), start=1):
    rid = row.get(ID_COL)
    if pd.isna(rid):
        continue
    rid = int(rid)

    if rid in ck_ids:
        continue

    q = build_query(pd.Series(row))
    out = {ID_COL: rid, "GEOCODE_QUERY": q}

    try:
        res = opencage_geocode(q)
        if res is None:
            out.update({"GEOCODE_STATUS": "no_match", "GEOCODE_REVIEW_FLAG": True})
        else:
            fields = extract_fields(res)
            out.update(fields)
            out["GEOCODE_STATUS"] = "ok"

            conf = fields.get("GEOCODE_CONFIDENCE")
            prov_in = row.get(PROVINCE_COL, "")
            prov_out = fields.get("GEOCODE_STATE", "")

            review = False
            if conf is None or float(conf) < CONFIDENCE_REVIEW_THRESHOLD:
                review = True
            if province_mismatch(prov_in, prov_out):
                review = True
            if fields.get("GEOCODE_LAT") is None or fields.get("GEOCODE_LON") is None:
                review = True

            out["GEOCODE_REVIEW_FLAG"] = review

    except Exception as e:
        out.update({"GEOCODE_STATUS": f"error: {type(e).__name__}", "GEOCODE_REVIEW_FLAG": True})

    buffer.append(out)
    new_done += 1

    if new_done % SAVE_EVERY == 0:
        tmp = pd.DataFrame(buffer)
        ck = tmp if ck.empty else pd.concat([ck, tmp], ignore_index=True)
        ck.to_csv(CHECKPOINT_PATH, index=False)
        ck_ids.update(tmp[ID_COL].astype(int).tolist())
        buffer = []
        print(f"Saved checkpoint after {new_done} new rows.")

    time.sleep(SLEEP_SECONDS)

# final flush
if buffer:
    tmp = pd.DataFrame(buffer)
    ck = tmp if ck.empty else pd.concat([ck, tmp], ignore_index=True)
    ck.to_csv(CHECKPOINT_PATH, index=False)

print("Done this run. New rows processed:", new_done)
print("Checkpoint:", CHECKPOINT_PATH)
ck2 = pd.read_csv(CHECKPOINT_PATH)
print(ck2["GEOCODE_STATUS"].value_counts(dropna=False))

Checkpoint found. Already processed: 2000
Saved checkpoint after 50 new rows.
Saved checkpoint after 100 new rows.
Saved checkpoint after 150 new rows.
Saved checkpoint after 200 new rows.
Saved checkpoint after 250 new rows.
Saved checkpoint after 300 new rows.
Saved checkpoint after 350 new rows.
Saved checkpoint after 400 new rows.
Saved checkpoint after 450 new rows.
Saved checkpoint after 500 new rows.
Saved checkpoint after 550 new rows.
Saved checkpoint after 600 new rows.
Saved checkpoint after 650 new rows.
Saved checkpoint after 700 new rows.
Saved checkpoint after 750 new rows.
Saved checkpoint after 800 new rows.
Saved checkpoint after 850 new rows.
Saved checkpoint after 900 new rows.
Saved checkpoint after 950 new rows.
Saved checkpoint after 1000 new rows.
Done this run. New rows processed: 1000
Checkpoint: C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\geocode_checkpoint_opencage.csv
ok          2538
no_match     462
Name: GEOCODE_STATUS, dtype: int